<a href="https://colab.research.google.com/github/AMLU-ANNA-JOSHY/Anomaly_Detection/blob/main/PaySim_fraud_detection_SQL_DNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fraud Detection on PaySim Dataset**

### **PaySim dataset from Kaggle**
- Synthetic dataset that resembles the normal operation of transactions and injects malicious behaviour for evaluating fraud detection methods.
- Simulates mobile money transactions and has fraud patterns like sudden large transfers and account draining.

Ref: https://www.kaggle.com/datasets/ealaxi/paysim1

In [ ]:
!pip install kaggle

In [ ]:
# Import necessary libraries
from google.colab import userdata
import os

# Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Retrieve secrets and write to kaggle.json
try:
    kaggle_username = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')

    with open('/root/.kaggle/kaggle.json', 'w') as f:
        f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')

    # Set appropriate permissions for the kaggle.json file
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API key configured successfully.")
except Exception as e:
    print(f"Error configuring Kaggle API key. Please check your Colab secrets: {e}")
    print("Make sure you have added 'KAGGLE_USERNAME' and 'KAGGLE_KEY' to Colab Secrets.")

In [ ]:
!kaggle datasets download -d ealaxi/paysim1

In [ ]:
!unzip paysim1.zip

In [5]:
import pandas as pd

df = pd.read_csv('PS_20174392719_1491204439457_log.csv')
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [6]:
# Drop irrelevant columns
df = df.drop(columns=["nameDest"])

# Understand fraud distribution
print(df['isFraud'].value_counts())

isFraud
0    6354407
1       8213
Name: count, dtype: int64


### **Setting up SQL**

In [7]:
import sqlite3

conn = sqlite3.connect("fraud.db")

print("Database created successfully")

Database created successfully


In [8]:
# Push contents of df into database

df.to_sql("transactions",
          conn, if_exists="replace", index=False)

6362620

In [ ]:
# inspect first contents

pd.read_sql('''
    SELECT *
    FROM transactions
    LIMIT 5;
    ''', conn)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,0.0,0.0,0,0


In [9]:
# rename for clarity

conn.execute("""
ALTER TABLE transactions
RENAME COLUMN nameOrig TO user_id;
""")

In [10]:
# create an index for direct look-up
# to identify rows faster in real time
# user_id -> sorted steps -> row locations
# Groups by user_id and sorts by step within group

conn.execute("""
CREATE INDEX idx_user_step
ON transactions(user_id, step);
""")

### **Extracting behavioural patterns using SQL**
- 1. Add txn_order to identify rows for each user, sorted by step: arrange all transactions by one user sorted by step(day), and assign a chronological number to it. => "position of transaction in user's history"
- 2. Apply moving window (sliding window) to get average amount of a batch of N transactions: **rolling average over the last N transactions per user** : understand what is the average amount spent by user: Normal user: spends around 200, Suddenly: 10,000 => possible fraud
- 3. Get **time and amount difference between transactions**: Normal: step: 1 ---- 10 ---- 20 ---- 40, Fraud: step: 100, 101, 102, 103 (very close) => burst

In [12]:
query = """
SELECT
    user_id,
    step,
    amount,
    oldbalanceOrg,
    newbalanceOrig,

    -- Transaction order
    ROW_NUMBER() OVER (
        PARTITION BY user_id ORDER BY step
    ) as txn_order,

    -- Rolling average (5 transactions)
    AVG(amount) OVER (
        PARTITION BY user_id
        ORDER BY step
        ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
    ) as avg_amt_5,

    -- Time gap
    step - LAG(step) OVER (
        PARTITION BY user_id ORDER BY step
    ) as step_diff,

    -- Amount change
    amount - LAG(amount) OVER (
        PARTITION BY user_id ORDER BY step
    ) as amt_diff,

    -- Balance difference
    (oldbalanceOrg - newbalanceOrig) as balance_diff,

    -- Fraud label
    isFraud

FROM transactions
"""
df_feat = pd.read_sql(query, conn)

In [13]:
# handle missing data

df_feat = df_feat.fillna(0)

In [14]:
# sort df

df_feat = df_feat.sort_values(["user_id", "step"])

In [ ]:
df_feat.head()

,user_id,step,amount,oldbalanceOrg,newbalanceOrig,txn_order,avg_amt_5,step_diff,amt_diff,balance_diff,isFraud
0,C1000000639,249,244486.46,8946.00,0.00,1,244486.46,0.0,0.0,8946.00,0
1,C1000001337,217,3170.28,58089.00,54918.72,1,3170.28,0.0,0.0,3170.28,0
2,C1000001725,46,8424.74,783.00,0.00,1,8424.74,0.0,0.0,783.00,0
3,C1000002591,231,261877.19,7596.00,269473.19,1,261877.19,0.0,0.0,-261877.19,0
4,C1000003372,167,20528.65,2302074.12,2322602.77,1,20528.65,0.0,0.0,-20528.65,0


In [15]:
import numpy as np

features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "avg_amt_5",
    "balance_diff",
    "step_diff",
    "amt_diff"
]


X = df_feat[features].values
y = df_feat["isFraud"].values

print(X.shape, y.shape)

(6362620, 7) (6362620,)


In [16]:
# partition data

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, stratify = y
)

### **XGBoost Classifier**

In [18]:
# to handle class imbalance add scale_pos_weight in XGBoostClassifier

scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
print(scale_pos_weight)

773.7482496194825


In [19]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [20]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [21]:
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00   1270881
           1       0.15      1.00      0.26      1643

    accuracy                           0.99   1272524
   macro avg       0.57      0.99      0.63   1272524
weighted avg       1.00      0.99      1.00   1272524

ROC-AUC: 0.9988478640939791


### **DNN Model for classification**

In [22]:
# convert to tensors

import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [23]:
# define dnn model

import torch.nn as nn

class DNNModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)

In [24]:
# instantiate model
model = DNNModel(input_size=len(features))

In [25]:
# understand data imbalance

from collections import Counter

counter = Counter(y_train.numpy())
print(counter)

Counter({np.float32(0.0): 5083526, np.float32(1.0): 6570})


In [26]:
# define loss weight to accomodate class imbalance

pos_weight = torch.tensor([counter[0] / counter[1]])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [27]:
# training loop

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()

    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 1933.3680
Epoch 2, Loss: 1179.7393
Epoch 3, Loss: 680.7050
Epoch 4, Loss: 669.2560
Epoch 5, Loss: 690.5389


In [36]:
from sklearn.metrics import roc_auc_score

model.eval()
with torch.no_grad():
    logits = model(X_test).squeeze()
    preds = torch.sigmoid(logits).numpy()

roc = roc_auc_score(y_test.numpy(), preds)
print("ROC-AUC:", roc)

ROC-AUC: 0.9293488030377179


**With an ROC_AUC score of 0.9988, XGBoost beats the result of DNN with 0.9293 score.**